# Session 5: Simulation, Parameter Recovery and Research Design
### Student Laboratory Workbook
*Course: Bayesian Analysis of Empirical Data (2026)*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/iknyazeva/bayes-cogsci-book/blob/main/notebooks/colab/05_simulation_parameter_recovery_research_design.ipynb)

---

## 1. Overview & Simulation-Based Research Design (SBRD)
Research design in modern Bayesian statistics is not a mechanical search for an "80% power" number against a null hypothesis. It is an active, prospective **computer simulation of your entire proposed study**:
1. **Simulate Before Collecting**: Generate synthetic datasets under realistic priors and noise before recruiting a single participant.
2. **Audit Parameter Recovery**: Verify that your estimator recovers true generating parameters without bias.
3. **Calibrate Intervals**: Confirm that 95% credible intervals achieve 95% empirical coverage.
4. **Beyond Power**: Quantify **Type S (Sign)** and **Type M (Magnitude Exaggeration)** errors under low signal-to-noise ratios.
5. **Optimize Allocation**: Evaluate whether resources are better spent adding participants ($J$) vs. repeated trials per participant ($K$).

---

### Structure of this Workbook:
* **Part I: Complete Worked Design Case — An Educational Intervention**: Step-by-step code and interactive plots simulating the classic *FakeMidtermFinal* trial (ROS Chapter 16).
* **Part II: Student Task — Suggest & Simulate Your Own Experiment**: Design your own study, set parameter truths, run recovery simulations, and produce decision operating curves.


In [1]:
# ==============================================================================
# 🚀 1. Setup Cell: Environment & Reproducibility Check
# ==============================================================================
import sys
import os
import numpy as np
import pandas as pd
from scipy import stats
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import matplotlib.pyplot as plt
import seaborn as sns

IS_COLAB = "google.colab" in sys.modules

if IS_COLAB:
    print("⚡ Running in Google Colab environment.")
    import plotly.io as pio
    pio.renderers.default = "colab"
else:
    print("💻 Running in local environment.")

RANDOM_SEED = 2026
rng = np.random.default_rng(RANDOM_SEED)
sns.set_style('whitegrid')
print(f"✅ Environment initialized. NumPy random seed set to {RANDOM_SEED}.")


💻 Running in local environment.
✅ Environment initialized. NumPy random seed set to 2026.


## 2. Observable Learning Targets
By completing this workbook, you will be able to:
1. **Build a Forward Data Generator**: Write modular `simulate_data()` functions encoding group assignments, latent parameters, and measurement noise.
2. **Execute a Recovery Loop**: Run $M = 1,000$ simulated experiments to evaluate bias, standard error, and empirical $95\%$ coverage.
3. **Construct Decision Operating Curves**: Plot sample size $N$ against uncertainty intervals and decision probabilities across different noise scenarios.
4. **Calculate Type S and Type M Errors**: Quantify the risk of sign reversals and effect size inflation when conditioning on statistical significance.
5. **Formulate Clustered Design Trade-Offs**: Optimize allocations between participants ($J$) and trials ($K$).
6. **Design and Simulate Your Own Experiment**: Formulate a complete prospective simulation plan for your research project.


---
## 3. Part I: Worked Design Case — An Educational Intervention
### Substantive Scenario (Adapted from Gelman, Hill, Vehtari 2021, *ROS* Ch. 16)
An educational researcher proposes a randomized controlled trial (RCT) to evaluate a new interactive tutoring intervention:
* **Target Population**: High school students taking a standardized final exam.
* **Control Group ($T_i = 0$)**: Standard classroom instruction. Mean score $\mu_0 = 60$ points, standard deviation $\sigma = 20$ points.
* **Treatment Group ($T_i = 1$)**: Interactive tutoring intervention. Hypothesized true improvement $\tau = +5$ points (i.e. mean score $\mu_1 = 65$).
* **Outcome Likelihood**:
  $$y_i \sim \operatorname{Normal}(\mu_0 + \tau \cdot T_i, \, \sigma^2), \quad \sigma = 20$$
* **Initial Proposed Design**: Total $N = 100$ students split evenly ($n_0 = 50$ control, $n_1 = 50$ treatment).

#### Theoretical Standard Error:
$$\text{SE}(\hat{\tau}) = \sqrt{\frac{\sigma^2}{n_0} + \frac{\sigma^2}{n_1}} = \sqrt{\frac{20^2}{50} + \frac{20^2}{50}} = \sqrt{8 + 8} = \mathbf{4.00} \text{ points}$$
* Approximate $95\%$ Confidence / Credible Interval Width: $2 \times 1.96 \times 4.00 \approx \mathbf{15.7} \text{ points}$.
* Notice that with an interval width of $15.7$, an interval around a true effect of $5.0$ will span from $[-2.8, +12.8]$, meaning a single study will frequently fail to exclude zero or even estimate a negative effect!


### Step 1: Single-Study Realization ($N = 100$)
Let us simulate what a single researcher would see if they ran this study once with $N = 100$ students:


In [2]:
# 1. Simulate one study of size N=100
N_single = 100
n_grp = N_single // 2
tau_true = 5.0
sigma_score = 20.0
mu_ctrl = 60.0

# Generate scores
scores_ctrl = rng.normal(loc=mu_ctrl, scale=sigma_score, size=n_grp)
scores_treat = rng.normal(loc=mu_ctrl + tau_true, scale=sigma_score, size=n_grp)

# Compute estimates
mean_c = scores_ctrl.mean()
mean_t = scores_treat.mean()
diff_hat = mean_t - mean_c
se_hat = np.sqrt(scores_ctrl.var(ddof=1)/n_grp + scores_treat.var(ddof=1)/n_grp)
ci_low, ci_high = diff_hat - 1.96 * se_hat, diff_hat + 1.96 * se_hat

print(f"=== SINGLE STUDY REALIZATION (N = {N_single}) ===")
print(f"Control Mean (n={n_grp}):   {mean_c:.2f} points (True: {mu_ctrl})")
print(f"Treatment Mean (n={n_grp}): {mean_t:.2f} points (True: {mu_ctrl + tau_true})")
print(f"Estimated Effect τ̂:         {diff_hat:.2f} points (True τ: {tau_true:.2f})")
print(f"Estimated Standard Error:   {se_hat:.2f} points (Theory: 4.00)")
print(f"95% Credible Interval:     [{ci_low:.2f}, {ci_high:.2f}] (Contains zero? {'YES' if ci_low <= 0 <= ci_high else 'NO'})")

# Plot individual scores with jitter
fig_single = go.Figure()

jit_c = rng.uniform(-0.15, 0.15, size=n_grp)
jit_t = 1.0 + rng.uniform(-0.15, 0.15, size=n_grp)

fig_single.add_trace(go.Scatter(x=jit_c, y=scores_ctrl, mode='markers', marker=dict(size=7, color='#94a3b8', opacity=0.7), name='Control Students'))
fig_single.add_trace(go.Scatter(x=jit_t, y=scores_treat, mode='markers', marker=dict(size=7, color='#2563eb', opacity=0.7), name='Treatment Students'))

fig_single.add_trace(go.Scatter(
    x=[0, 1], y=[mean_c, mean_t],
    mode='markers+lines',
    marker=dict(size=12, color='#0f172a', symbol='diamond'),
    line=dict(color='#0f172a', width=2),
    name='Sample Means',
    error_y=dict(type='data', array=[1.96 * (scores_ctrl.std()/np.sqrt(n_grp)), 1.96 * (scores_treat.std()/np.sqrt(n_grp))], visible=True)
))

fig_single.update_layout(
    title=f'Single Study Sample (N = {N_single}): Estimated Effect τ̂ = {diff_hat:.2f} pts [95% CI: {ci_low:.1f}, {ci_high:.1f}]',
    xaxis=dict(tickvals=[0, 1], ticktext=['Control (n=50)', 'Treatment (n=50)'], title='Experimental Group'),
    yaxis_title='Final Exam Score (Points)',
    template='plotly_white',
    height=420
)
fig_single.show()


=== SINGLE STUDY REALIZATION (N = 100) ===
Control Mean (n=50):   64.34 points (True: 60.0)
Treatment Mean (n=50): 62.46 points (True: 65.0)
Estimated Effect τ̂:         -1.88 points (True τ: 5.00)
Estimated Standard Error:   4.06 points (Theory: 4.00)
95% Credible Interval:     [-9.83, 6.07] (Contains zero? YES)


### Step 2: Repeated Studies Simulation ($M = 1,000$ Replications)
Never judge a study design by one realization! We now simulate $M = 1,000$ independent replications of this exact same $N = 100$ study to examine the **sampling distribution** and **operating characteristics**.


In [3]:
M_sims = 1000

# Simulate M experiments
ctrl_all = rng.normal(loc=mu_ctrl, scale=sigma_score, size=(M_sims, n_grp))
treat_all = rng.normal(loc=mu_ctrl + tau_true, scale=sigma_score, size=(M_sims, n_grp))

tau_estimates = treat_all.mean(axis=1) - ctrl_all.mean(axis=1)
se_theoretical = np.sqrt(2 * (sigma_score**2) / n_grp)

ci_lows = tau_estimates - 1.96 * se_theoretical
ci_highs = tau_estimates + 1.96 * se_theoretical

coverage_rate = np.mean((ci_lows <= tau_true) & (tau_true <= ci_highs))
prop_positive = np.mean(tau_estimates > 0)
prop_significant = np.mean(ci_lows > 0)

print(f"=== OPERATING CHARACTERISTICS ACROSS {M_sims:,} REPETITIONS (N = {N_single}) ===")
print(f"True Treatment Effect (τ):      {tau_true:.2f} points")
print(f"Average Estimated Effect (E[τ̂]): {tau_estimates.mean():.2f} points (Bias = {tau_estimates.mean() - tau_true:.3f})")
print(f"Standard Deviation of Estimates: {tau_estimates.std():.2f} points (Theoretical SE = {se_theoretical:.2f})")
print(f"Empirical 95% Coverage Rate:    {coverage_rate*100:.1f}% (Nominal Target = 95.0%)")
print(f"Probability of Correct Sign:    {prop_positive*100:.1f}%")
print(f"Statistical Power (p < 0.05):   {prop_significant*100:.1f}%")

# Plot sampling distribution
x_axis = np.linspace(-10, 20, 300)
pdf_theory = stats.norm.pdf(x_axis, loc=tau_true, scale=se_theoretical)

fig_m = go.Figure()
fig_m.add_trace(go.Histogram(
    x=tau_estimates, histnorm='probability density',
    marker=dict(color='#3b82f6', opacity=0.75, line=dict(color='#1d4ed8', width=1)),
    name='Simulated τ̂ Estimates'
))
fig_m.add_trace(go.Scatter(
    x=x_axis, y=pdf_theory, mode='lines',
    line=dict(color='#1e40af', width=2.5),
    name=f'Theoretical Gaussian (SE={se_theoretical:.1f})'
))
fig_m.add_vline(x=tau_true, line_dash='dash', line_color='#d97706', annotation_text='True τ = +5.0')
fig_m.add_vline(x=0, line_dash='dot', line_color='#dc2626', annotation_text='Zero Effect (Null)')

fig_m.update_layout(
    title=f'Sampling Distribution of Treatment Effect τ̂ across {M_sims} Studies (N = {N_single}, Power = {prop_significant*100:.1f}%)',
    xaxis_title='Estimated Treatment Effect τ̂ (Points)',
    yaxis_title='Probability Density',
    template='plotly_white',
    height=420
)
fig_m.show()


=== OPERATING CHARACTERISTICS ACROSS 1,000 REPETITIONS (N = 100) ===
True Treatment Effect (τ):      5.00 points
Average Estimated Effect (E[τ̂]): 5.04 points (Bias = 0.037)
Standard Deviation of Estimates: 4.02 points (Theoretical SE = 4.00)
Empirical 95% Coverage Rate:    94.7% (Nominal Target = 95.0%)
Probability of Correct Sign:    90.1%
Statistical Power (p < 0.05):   23.5%


### Step 3: Sample Size Scaling ($N = 100 \to 400 \to 900$)
What happens when the school district expands the study?


In [4]:
sample_sizes = [100, 400, 900]
summary_rows = []

for N_val in sample_sizes:
    n_g = N_val // 2
    se_val = np.sqrt(2 * (sigma_score**2) / n_g)
    ci_w = 2 * 1.96 * se_val
    power_val = (1.0 - stats.norm.cdf(1.96 - tau_true / se_val)) * 100
    
    summary_rows.append({
        'Total Students (N)': N_val,
        'Students / Group': n_g,
        'Standard Error (SE)': f"{se_val:.2f} pts",
        '95% CI Width': f"{ci_w:.1f} pts",
        'Typical 95% Interval': f"[{tau_true - 1.96*se_val:.1f}, {tau_true + 1.96*se_val:.1f}]",
        'Statistical Power': f"{power_val:.1f}%"
    })

df_scaling = pd.DataFrame(summary_rows)
print("=== SAMPLE SIZE SCALING SUMMARY TABLE ===")
display(df_scaling)


=== SAMPLE SIZE SCALING SUMMARY TABLE ===


,Total Students (N),Students / Group,Standard Error (SE),95% CI Width,Typical 95% Interval,Statistical Power
0,100,50,4.00 pts,15.7 pts,"[-2.8, 12.8]",23.9%
1,400,200,2.00 pts,7.8 pts,"[1.1, 8.9]",70.5%
2,900,450,1.33 pts,5.2 pts,"[2.4, 7.6]",96.3%


### Step 4: Decision Operating Curves across Sample Size and Noise
Instead of relying on a single power number, we construct **operating curves** showing how precision and power scale across sample size $N \in [40, 1000]$ and residual noise levels $\sigma \in \{10, 20, 30\}$.


In [5]:
N_range = np.arange(40, 1001, 20)
sigmas = [10.0, 20.0, 30.0]
colors_sigma = ['#10b981', '#3b82f6', '#ef4444']

fig_curves = make_subplots(
    rows=1, cols=2,
    subplot_titles=[
        '<b>(a) Precision: 95% Credible Interval Width</b><br><span style="font-size:11px;color:#64748b">Smaller is more precise (Target < 5 points)</span>',
        '<b>(b) Statistical Power: P(Reject Zero at α=0.05)</b><br><span style="font-size:11px;color:#64748b">Probability of discovering true effect τ = +5</span>'
    ]
)

for sigma_val, col in zip(sigmas, colors_sigma):
    se_vals = np.sqrt(2 * (sigma_val**2) / (N_range / 2))
    ci_widths = 2 * 1.96 * se_vals
    powers = (1.0 - stats.norm.cdf(1.96 - tau_true / se_vals)) * 100
    
    fig_curves.add_trace(go.Scatter(
        x=N_range, y=ci_widths, mode='lines',
        line=dict(color=col, width=2.5),
        name=f'σ = {sigma_val:.0f} pts'
    ), row=1, col=1)
    
    fig_curves.add_trace(go.Scatter(
        x=N_range, y=powers, mode='lines',
        line=dict(color=col, width=2.5),
        name=f'σ = {sigma_val:.0f} pts',
        showlegend=False
    ), row=1, col=2)

# Target line for power 80%
fig_curves.add_hline(y=80, line_dash='dash', line_color='#475569', annotation_text='80% Power', row=1, col=2)
fig_curves.add_hline(y=5, line_dash='dash', line_color='#475569', annotation_text='5 pt Interval Width', row=1, col=1)

fig_curves.update_layout(template='plotly_white', height=420)
fig_curves.update_xaxes(title_text='Total Sample Size N')
fig_curves.update_yaxes(title_text='95% Interval Width (Points)', row=1, col=1)
fig_curves.update_yaxes(title_text='Power (%)', row=1, col=2)
fig_curves.show()


### Step 5: Beyond Power — Type S and Type M Errors (*Gelman & Carlin 2014*)
When power is low (e.g. $N = 100$, power $\approx 26\%$), filtering on $p < 0.05$ creates two severe distortions:
1. **Type S (Sign) Error**: The probability that a statistically significant finding has the **wrong sign** (points negative).
2. **Type M (Magnitude) Exaggeration Factor**: The average factor by which significant findings **overestimate** the true effect size.


In [6]:
def compute_type_s_m(tau, se, alpha=0.05):
    z_crit = stats.norm.ppf(1 - alpha / 2)
    # Probability of statistically significant positive estimate
    p_pos = 1 - stats.norm.cdf(z_crit - tau / se)
    # Probability of statistically significant negative estimate (Type S)
    p_neg = stats.norm.cdf(-z_crit - tau / se)
    total_power = p_pos + p_neg
    
    type_s = p_neg / total_power if total_power > 0 else 0
    
    # Expected value of |theta_hat| given significance
    # Using truncated normal integration
    grid = np.linspace(-6 * se, 6 * se, 5000)
    pdf = stats.norm.pdf(grid, loc=tau, scale=se)
    sig_mask = np.abs(grid / se) > z_crit
    expected_mag = np.sum(np.abs(grid)[sig_mask] * pdf[sig_mask]) / np.sum(pdf[sig_mask])
    type_m = expected_mag / abs(tau)
    
    return total_power, type_s, type_m

power_100, type_s_100, type_m_100 = compute_type_s_m(tau=5.0, se=4.0)
power_400, type_s_400, type_m_400 = compute_type_s_m(tau=5.0, se=2.0)
power_900, type_s_900, type_m_900 = compute_type_s_m(tau=5.0, se=1.33)

print("=== TYPE S AND TYPE M ERROR ANALYSIS ===")
print(f"N = 100 (SE = 4.0): Power = {power_100*100:.1f}%, Type S Error = {type_s_100*100:.2f}%, Exaggeration Factor = {type_m_100:.2f}x")
print(f"N = 400 (SE = 2.0): Power = {power_400*100:.1f}%, Type S Error = {type_s_400*100:.4f}%, Exaggeration Factor = {type_m_400:.2f}x")
print(f"N = 900 (SE = 1.3): Power = {power_900*100:.1f}%, Type S Error = 0.00%, Exaggeration Factor = {type_m_900:.2f}x")


=== TYPE S AND TYPE M ERROR ANALYSIS ===
N = 100 (SE = 4.0): Power = 24.0%, Type S Error = 0.28%, Exaggeration Factor = 2.04x
N = 400 (SE = 2.0): Power = 70.5%, Type S Error = 0.0006%, Exaggeration Factor = 1.19x
N = 900 (SE = 1.3): Power = 96.4%, Type S Error = 0.00%, Exaggeration Factor = 1.01x


### Step 6: Clustered Allocation — Participants ($J$) vs. Trials ($K$)
In cognitive science and education, observations are often nested (e.g. $K$ test trials per student, or $K$ students per school):
$$\text{SE}(\hat{\mu}) = \sqrt{\frac{\sigma_{\text{person}}^2}{J} + \frac{\sigma_{\text{trial}}^2}{J \times K}}$$
* As $K \to \infty$, the standard error hits an impassable floor: $\frac{\sigma_{\text{person}}}{\sqrt{J}}$.
* Adding more trials $K$ has rapidly diminishing returns; recruiting more participants $J$ is the only way to break through the precision ceiling.


In [7]:
# Heatmap: Standard Error across J participants and K trials
J_vals = np.arange(10, 101, 5)
K_vals = np.arange(5, 101, 5)
J_grid, K_grid = np.meshgrid(J_vals, K_vals)

sigma_p = 15.0  # Between-person variation
sigma_t = 20.0  # Within-person trial variation

se_matrix = np.sqrt((sigma_p**2)/J_grid + (sigma_t**2)/(J_grid * K_grid))

fig_alloc = go.Figure(go.Heatmap(
    x=J_vals, y=K_vals, z=se_matrix,
    colorscale='Viridis', reversescale=True,
    colorbar=dict(title='SE of Mean')
))
fig_alloc.update_layout(
    title='Design Allocation: Participants (J) vs. Trials per Participant (K)',
    xaxis_title='Number of Participants (J)',
    yaxis_title='Number of Trials per Participant (K)',
    template='plotly_white',
    height=440
)
fig_alloc.show()


---
## 4. Part II: Student Assignment — Suggest & Simulate Your Own Experiment

Now it is your turn! You will formulate a prospective simulation-based design analysis for a study of your choice.

### 📋 Assignment Instructions:
1. **Choose Your Substantive Scientific Domain**:
   * *Example A (Cognitive Psychology)*: Does a 10-minute mindfulness induction reduce reaction time in a Stroop interference task?
   * *Example B (Digital Health / Clinical)*: Does a mobile notification app reduce self-reported daily stress scores?
   * *Example C (Educational Interventions)*: Does automated AI feedback improve essay writing scores compared to self-editing?
   * *Example D (Consumer / Social Decision-Making)*: Does a transparent carbon label increase the probability of choosing sustainable products?
2. **Declare Your Formal Model Parameters**:
   * Baseline control mean ($\mu_0$) or baseline success rate ($\theta_0$).
   * Hypothesized true treatment effect ($\tau_{\text{true}}$).
   * Residual measurement noise ($\sigma$).
3. **Run the Simulation Sandbox Below**:
   * Test at least 3 candidate sample sizes (e.g. $N = 30$, $N = 100$, $N = 300$).
   * Evaluate the sampling distribution, empirical coverage, statistical power, and Type M exaggeration factor.
4. **Write Your Design Recommendation in the Reflection Cell**:
   * Which sample size do you recommend and why?


In [8]:
# ==============================================================================
# 🧪 STUDENT EXPERIMENT SANDBOX: Plug In Your Own Parameters
# ==============================================================================

# 1. State your study parameters:
my_experiment_name = "Cognitive Working Memory Training"
my_mu0 = 50.0          # Baseline control score (e.g. 50 pts)
my_tau_true = 4.0      # Hypothesized treatment effect (+4 pts)
my_sigma = 12.0        # Residual measurement noise (SD = 12 pts)

# 2. Candidate sample sizes to test:
my_sample_sizes = [40, 100, 250]
my_replications = 1000

my_results = []

for N_cand in my_sample_sizes:
    n_half = N_cand // 2
    # Simulate M studies
    c_draws = rng.normal(loc=my_mu0, scale=my_sigma, size=(my_replications, n_half))
    t_draws = rng.normal(loc=my_mu0 + my_tau_true, scale=my_sigma, size=(my_replications, n_half))
    
    diffs = t_draws.mean(axis=1) - c_draws.mean(axis=1)
    se_th = np.sqrt(2 * (my_sigma**2) / n_half)
    
    # Metrics
    power_est, type_s_est, type_m_est = compute_type_s_m(my_tau_true, se_th)
    ci_w_est = 2 * 1.96 * se_th
    
    my_results.append({
        'Sample Size (N)': N_cand,
        'SE': f"{se_th:.2f}",
        '95% CI Width': f"{ci_w_est:.2f}",
        'Power (%)': f"{power_est*100:.1f}%",
        'Type S Error (%)': f"{type_s_est*100:.2f}%",
        'Type M Exaggeration': f"{type_m_est:.2f}x"
    })

df_my_study = pd.DataFrame(my_results)
print(f"=== DESIGN ANALYSIS FOR: {my_experiment_name.upper()} ===")
print(f"True Effect: τ = {my_tau_true}, Noise: σ = {my_sigma}")
display(df_my_study)


=== DESIGN ANALYSIS FOR: COGNITIVE WORKING MEMORY TRAINING ===
True Effect: τ = 4.0, Noise: σ = 12.0


,Sample Size (N),SE,95% CI Width,Power (%),Type S Error (%),Type M Exaggeration
0,40,3.79,14.88,18.4%,0.70%,2.37x
1,100,2.40,9.41,38.5%,0.04%,1.60x
2,250,1.52,5.95,75.0%,0.00%,1.16x


---
## 5. Exit Record & Design Reflection
> ✍ **WRITE (Your Design Evaluation)**:
> 1. **Substantive Question**: What is your proposed research question and target estimand?
> 2. **Signal-to-Noise Ratio**: What is $\tau_{\text{true}} / \sigma$? Is this a small, medium, or large effect?
> 3. **Design Choice**: Based on your simulation table above, which sample size is adequate? Why is the smallest sample size ($N = 40$) risky in terms of Type M exaggeration?
> 4. **Limitation**: What real-world complication (e.g. attrition, compliance, clustering) did your simple simulation ignore?


---
## 6. Reproducibility Footer
* Course: Bayesian Analysis of Empirical Data (2026)
* Session 5 Laboratory: Simulation-Based Research Design
* Validated in Google Colab and local Python 3.12 environments
